# Day 15 — Memory Model & a Bump Allocator, Visualized

A process's address space has a **stack** (grows down, automatic) and a **heap** (grows
up, you manage it). A bump allocator carves the heap by advancing a cursor — fast, and
allocation-free at steady state.

## A process's address space (schematically)

```
  high addr  |  stack   |  <- grows DOWN (locals, call frames)
             |    |     |
             |    v     |
             |          |
             |    ^     |
             |    |     |
             |  heap    |  <- grows UP (malloc / your allocator)
  low addr   |  code    |
```

In [ ]:
class Bump:
    def __init__(self, cap): self.cap = cap; self.off = 0; self.blocks = []
    def alloc(self, size, align=1):
        start = ((self.off + align - 1) // align) * align
        if start + size > self.cap: raise MemoryError('OOM')
        self.blocks.append((self.off, start, size))  # (padding_start, block_start, size)
        self.off = start + size; return start

a = Bump(64)
a.alloc(3)          # 3 bytes
a.alloc(8, align=8) # aligned to 8 -> padding inserted
a.alloc(4, align=4)
print('blocks (pad_start, start, size):', a.blocks)
print('used:', a.off, 'of', a.cap)

## The arena, drawn (blue = data, red = alignment padding)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 1.6))
for pad_start, start, size in a.blocks:
    if start > pad_start:
        ax.barh(0, start - pad_start, left=pad_start, color='#DD8452')  # padding
    ax.barh(0, size, left=start, color='#4C72B0', edgecolor='white')
    ax.text(start + size / 2, 0, str(size), ha='center', va='center', color='white')
ax.barh(0, a.cap - a.off, left=a.off, color='#dddddd')                  # free
ax.set_xlim(0, a.cap); ax.set_yticks([]); ax.set_title('bump arena layout'); plt.show()

## Takeaways

- Stack = automatic LIFO; heap = you manage lifetime.
- A bump allocator is O(1) and allocation-free at steady state — great for real-time loops.
- Alignment costs a little padding but the hardware needs it.

**Now build it** (alloc + reset, with OOM handling) in `homework.py`, then `pytest -q`.